# January 2025 BTS data: a first look

I wanted to get a feel for the file before deciding what is worth modelling. These are small checks and questions, not a complete analysis. Some aircraft checks are only clues. BTS clock fields are local time, and a tail number in the history does not tell us whether it would have been known when a prediction was made.

First create the lightly cleaned working file with `.\.venv\Scripts\python.exe scripts\prepare_january_sample.py` from the repository root. This notebook reads that local file. It does not change the raw data or train a model.

## First, what is in this file?

I am starting with a handful of fields so it is easier to inspect the data and missing values.

In [3]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA_FILE = ROOT / 'data' / 'processed' / 'january_2025_clean.csv'

cols = ['FlightDate', 'Reporting_Airline', 'Tail_Number', 'Origin', 'Dest',
        'DepDelay', 'ArrDelay', 'CRSDepTime', 'CRSArrTime', 'DepTime',
        'ArrTime', 'Cancelled', 'Diverted']
df = pd.read_csv(DATA_FILE, usecols=cols, low_memory=False)
df['FlightDate'] = pd.to_datetime(df['FlightDate'], errors='coerce')
df['Tail_Number'] = df['Tail_Number'].astype('string').str.strip().str.upper()
df['Tail_Number'] = df['Tail_Number'].mask(df['Tail_Number'].eq(''))
print('File:', DATA_FILE)
print('Rows and columns:', df.shape)
print('Dates:', df['FlightDate'].min(), 'to', df['FlightDate'].max())
df.head()

File: C:\Users\subed\Documents\flight-delay-cascade\data\processed\january_2025_clean.csv
Rows and columns: (539747, 13)
Dates: 2025-01-01 00:00:00 to 2025-01-31 00:00:00


  FlightDate Reporting_Airline Tail_Number Origin Dest  CRSDepTime  ...  DepDelay  CRSArrTime  ArrTime  ArrDelay  Cancelled  Diverted
0 2025-01-01                AA      N104NN    JFK  LAX         659  ...      -3.0        1020   1013.0      -7.0        0.0       0.0
1 2025-01-02                AA      N110AN    JFK  LAX         659  ...      -7.0        1020   1022.0       2.0        0.0       0.0
2 2025-01-03                AA      N106NN    JFK  LAX         659  ...      -7.0        1020   1003.0     -17.0        0.0       0.0
3 2025-01-04                AA      N117AN    JFK  LAX         700  ...      -7.0        1026   1016.0     -10.0        0.0       0.0
4 2025-01-05                AA      N104NN    JFK  LAX         659  ...      -4.0        1020   1013.0      -7.0        0.0       0.0

The saved check found 539,747 rows covering January 1 to 31. Next I want to see whether missing values are concentrated in particular fields or flight statuses.

In [5]:
missing = df[cols].isna().sum().sort_values(ascending=False)
missing.to_frame('missing rows').assign(percent=lambda x: (100 * x['missing rows'] / len(df)).round(2))

                   missing rows  percent
ArrDelay                  17478     3.24
ArrTime                   16580     3.07
DepDelay                  15923     2.95
DepTime                   15886     2.94
Tail_Number                2530     0.47
Dest                          0     0.00
Origin                        0     0.00
Reporting_Airline             0     0.00
FlightDate                    0     0.00
CRSArrTime                    0     0.00
CRSDepTime                    0     0.00
Cancelled                     0     0.00
Diverted                      0     0.00

## How are the delay values distributed?

Negative arrival delay means the flight arrived early. I am looking at a few quantiles and the largest values to catch anything that needs a closer look. These are observed outcomes, so they are not candidate model inputs.

In [7]:
arr = df['ArrDelay'].dropna()
arr.describe(percentiles=[.10, .25, .50, .75, .90, .95, .99]).to_frame('ArrDelay minutes')

# Inspect the largest recorded values in case they reflect unusual events or data issues.
df.nlargest(10, 'ArrDelay')[['FlightDate', 'Reporting_Airline', 'Origin', 'Dest', 'ArrDelay']]

       FlightDate Reporting_Airline Origin Dest  ArrDelay
57434  2025-01-28                AA    IAH  DFW    3282.0
502118 2025-01-03                MQ    MLU  DFW    2955.0
5333   2025-01-09                AA    JAC  ORD    2680.0
10583  2025-01-13                AA    CHS  CLT    2448.0
46288  2025-01-15                AA    SDF  CLT    2310.0
18614  2025-01-23                AA    RDU  PHL    2197.0
21553  2025-01-15                AA    MFE  DFW    2157.0
67391  2025-01-15                AA    STL  CLT    2065.0
6139   2025-01-20                AA    SDF  CLT    1856.0
513009 2025-01-19                MQ    AVP  ORD    1843.0

The existing summary gives a median of -8 minutes and a 90th percentile of 37 minutes, among rows with an arrival delay. I also want a simple daily view. A rough pattern in one month is something to investigate, not proof of a trend.

In [9]:
daily = df.groupby('FlightDate').agg(
    flights=('FlightDate', 'size'),
    delay_values=('ArrDelay', 'count'),
    median_arrival_delay=('ArrDelay', 'median'),
    p90_arrival_delay=('ArrDelay', lambda x: x.quantile(.90)),
    delayed_15_or_more=('ArrDelay', lambda x: x.ge(15).sum()),
)
daily.head(10)

            flights  delay_values  median_arrival_delay  p90_arrival_delay  delayed_15_or_more
FlightDate                                                                                    
2025-01-01    16827         16771                  -8.0               25.0                2566
2025-01-02    19640         19431                  -7.0               30.0                3285
2025-01-03    19358         19228                  -3.0               54.0                5165
2025-01-04    18544         18257                  -1.0               57.0                5458
2025-01-05    19676         18151                   0.0               84.0                6014
2025-01-06    18732         16816                   5.0              101.0                6752
2025-01-07    15974         15445                  -5.0               50.0                3877
2025-01-08    16276         16071                  -9.0               25.0                2373
2025-01-09    17952         15937                 

The daily table caught my eye: the 90th percentile is much higher on January 5 and 6 than on several nearby days. I want to see whether that shows up across the board or is concentrated in a few airports. It is a lead to check, not an explanation.

In [11]:
busy_days = df[df['FlightDate'].isin(pd.to_datetime(['2025-01-05', '2025-01-06']))]
airport_check = busy_days.groupby('Origin').agg(
    flights=('Origin', 'size'),
    delay_values=('ArrDelay', 'count'),
    median_delay=('ArrDelay', 'median'),
    p90_delay=('ArrDelay', lambda x: x.quantile(.90)),
).query('delay_values >= 100').sort_values('p90_delay', ascending=False)
airport_check.head(10).round(1)

        flights  delay_values  median_delay  p90_delay
Origin                                                
STL         342           111         102.0      232.0
BWI         526           367          19.0      214.0
CVG         179           106          25.5      205.5
DFW        1784          1544          36.0      186.7
IAD         284           219          26.0      170.2
IAH         633           584          12.0      167.7
PHL         484           436          20.0      159.5
OKC         125           115          15.0      134.6
MKE         133           120           7.0      125.7
ORD        1530          1360          14.5      122.1

STL stands out in this two-day check, with a 232-minute 90th percentile from 111 flights with a recorded arrival delay. That is a reason to inspect those records and dates more closely. It does not tell me what caused the delays, and it is a much smaller group than the largest airports.

## Do airlines and airports look very different?

These tables are just a first comparison. The busiest places have more rows, and these raw summaries do not account for routes, schedules, weather, or other differences.

In [14]:
airline = df.groupby('Reporting_Airline').agg(
    records=('ArrDelay', 'size'),
    delay_values=('ArrDelay', 'count'),
    median_delay=('ArrDelay', 'median'),
    percent_15_plus=('ArrDelay', lambda x: 100 * x.dropna().ge(15).mean()),
).sort_values('records', ascending=False)
airline.head(15).round(1)

                   records  delay_values  median_delay  percent_15_plus
Reporting_Airline                                                      
WN                  105307        102120          -9.0             15.2
DL                   76306         74025          -9.0             19.3
AA                   75088         72082          -8.0             19.2
OO                   65036         63502          -8.0             20.9
UA                   62007         60668          -8.0             17.6
YX                   27833         26683         -11.0             16.0
MQ                   21890         20831          -8.0             18.4
OH                   21094         19151          -5.0             25.5
AS                   18163         17847          -7.0             17.6
B6                   17918         17558          -6.0             24.5
NK                   17544         16946          -9.0             18.1
F9                   15526         15110          -6.0          

In [15]:
origin = df.groupby('Origin').agg(
    departures=('Origin', 'size'),
    delay_values=('ArrDelay', 'count'),
    median_arrival_delay=('ArrDelay', 'median'),
).sort_values('departures', ascending=False)
origin.head(15).round(1)

        departures  delay_values  median_arrival_delay
Origin                                                
DFW          25124         23469                  -7.0
DEN          24732         24350                  -5.0
ATL          23881         22580                  -4.0
ORD          21643         21200                  -9.0
CLT          16877         16083                  -6.0
PHX          15956         15770                 -10.0
LAX          15157         14958                 -12.0
LAS          15007         14803                 -11.0
MCO          13058         12817                  -7.0
SEA          11882         11719                  -9.0
DCA          11734         10619                  -6.0
LGA          10905         10566                  -9.0
SFO          10839         10720                 -10.0
BOS          10624         10356                  -9.0
MIA          10182          9939                  -6.0

## What happens to records without an arrival delay?

I noticed missing arrival delays in the first summary. Check whether those records are cancelled or diverted before deciding how to handle them.

In [17]:
cancelled = df['Cancelled'].eq(1)
diverted = df['Diverted'].eq(1)
no_arrival_delay = df['ArrDelay'].isna()
pd.Series({
    'cancelled': cancelled.sum(),
    'diverted': diverted.sum(),
    'missing arrival delay': no_arrival_delay.sum(),
    'missing and cancelled': (no_arrival_delay & cancelled).sum(),
    'missing and diverted': (no_arrival_delay & diverted).sum(),
    'missing, neither cancelled nor diverted': (no_arrival_delay & ~cancelled & ~diverted).sum(),
}, name='rows')

cancelled                                  16312
diverted                                    1166
missing arrival delay                      17478
missing and cancelled                      16312
missing and diverted                        1166
missing, neither cancelled nor diverted        0

In this month, the 17,478 missing arrival delays match the cancellation and diversion records. That is useful for understanding this sample, but I would recheck it on any additional months before relying on it.

## Are repeated tail numbers enough to build aircraft links?

Not by themselves. I first count repeated tail/date groups, then inspect a few examples. Local clock times and overnight flights make sorting by date and clock time an imperfect way to reconstruct the real sequence.

In [20]:
operated = df.loc[~cancelled & ~diverted & df['Tail_Number'].notna()].copy()
tail_day = operated.groupby(['Tail_Number', 'FlightDate']).size().rename('flights')
print('Tail/date groups:', len(tail_day))
print('Groups with more than one record:', tail_day.gt(1).sum())
print('Largest group:', tail_day.max())
tail_day[tail_day.gt(1)].sort_values(ascending=False).head(12)

Tail/date groups: 142174
Groups with more than one record: 127981
Largest group: 13


Tail_Number  FlightDate
N478HA       2025-01-13    13
N480HA       2025-01-23    13
N478HA       2025-01-24    13
N480HA       2025-01-16    13
N488HA       2025-01-31    13
N490HA       2025-01-27    13
N489HA       2025-01-09    13
N478HA       2025-01-23    12
N477HA       2025-01-02    12
             2025-01-07    12
N494HA       2025-01-07    12
             2025-01-24    12

In [21]:
# Pick one busy tail/date group to inspect. This is an example, not a validated rotation.
example_key = tail_day[tail_day.ge(4)].index[0]
example = operated.loc[
    operated['Tail_Number'].eq(example_key[0]) & operated['FlightDate'].eq(example_key[1]),
    ['FlightDate', 'Tail_Number', 'Origin', 'Dest', 'CRSDepTime', 'CRSArrTime', 'DepTime', 'ArrTime', 'DepDelay', 'ArrDelay']
].sort_values('CRSDepTime')
example

       FlightDate Tail_Number Origin Dest  CRSDepTime  CRSArrTime  DepTime  ArrTime  DepDelay  ArrDelay
487122 2025-01-02       188NV    FLL  LEX         600         828    557.0    807.0      -3.0     -21.0
490635 2025-01-02       188NV    LEX  FLL         918        1145    909.0   1124.0      -9.0     -21.0
493147 2025-01-02       188NV    FLL  USA        1235        1435   1231.0   1436.0      -4.0       1.0
493638 2025-01-02       188NV    USA  FLL        1525        1726   1542.0   1741.0      17.0      15.0
488648 2025-01-02       188NV    FLL  TYS        1816        2020   1844.0   2051.0      28.0      31.0
490174 2025-01-02       188NV    TYS  FLL        2110        2320   2142.0      5.0      32.0      45.0

Question to carry forward: after correcting for local airport dates and time zones, can candidate consecutive legs be checked against actual arrival and departure times? Historical tail assignments also need a separate availability check before they can be features in a prediction.

## What I would check next

- Repeat these checks with another month so January is not treated as representative of a whole year.
- Inspect candidate aircraft legs around local midnight and where one flight has several possible successors.
- Agree on the prediction target and how far before departure the prediction is meant to be made.
- Check that every proposed input would actually be available at that time.

I have not trained a model here. The point of this first look is to decide which data questions to settle before training.